# Contrastive Activation Addition (CAA)

**Paper.** [Steering Llama 2 via Contrastive Activation Addition](https://arxiv.org/abs/2312.06681)

**Authors.** Nina Panickssery, Nick Gabrieli, Julian Schulz, Meg Tong, Evan Hubinger, Alexander Matt Turner

CAA is a state control method that steers model behavior by adding a learned direction vector to the residual stream during generation. The direction is the mean difference between hidden states on paired examples that do and do not exhibit a target behavior. At inference time the vector is added at a single layer with a configurable `multiplier`, so the sign and magnitude of the `multiplier` set the direction and degree of steering.

The same mean-difference extraction is the core of recent work on trait steering. [Persona Vectors: Monitoring and Controlling Character Traits in Language Models](https://arxiv.org/abs/2507.21509) fits directions for character traits by contrasting activations on responses that exhibit the trait against responses that do not, and uses them to monitor and steer chat models. This notebook applies CAA to a persona dimension of that kind, i.e., formality, the axis running from a casual register to a formal one. A single fitted direction serves both ends of the axis, with the sign of `multiplier` selecting the direction of steering. We fit a formality direction for `ibm-granite/granite-4.1-3b` from a small pool of contrastive responses, steer the register in both directions on held-out prompts, then save the fitted vector and reuse it with different steering parameters, first in process on the Hugging Face backend and then through a vLLM server.

## Method parameters

| parameter | type | description |
| --- | --- | --- |
| `steering_vector` | `SteeringVector` | Pre-computed steering vector, used instead of `data` |
| `data` | `ContrastivePairs` | Paired positive/negative texts used to fit the vector during `steer()` |
| `train_spec` | `VectorTrainSpec` | Extraction config (method, accumulation mode, rendering, boundary) |
| `layer_id` | `int` | Layer the vector is added at. `None` selects a layer at roughly 40 percent depth |
| `multiplier` | `float` | Scale on the vector. Positive increases the target behavior, negative decreases it |
| `token_scope` | `str` | Positions the vector is added at: `"after_prompt"` (default), `"all"`, `"last_k"`, or `"from_position"` |
| `last_k` | `int` | Number of trailing positions when `token_scope="last_k"` |
| `from_position` | `int` | Absolute start position when `token_scope="from_position"` |
| `normalize_vector` | `bool` | L2-normalize the vector before applying |
| `use_norm_preservation` | `bool` | Rescale steered hidden states whose norm increased back to their pre-steering norm |

## Setup

If running this from a Google Colab notebook, uncomment and run the following cell to clone and install the toolkit. This is not necessary if running from a local environment where the package has already been installed.

In [ ]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360
# !pip install -q -e .

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub.

In [ ]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

In [ ]:
import sys
!{sys.executable} -m pip install -q tabulate

In [ ]:
import os
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

from aisteer360.algorithms.core.execution import BackendSpec
from aisteer360.algorithms.core.internals import ContrastivePairs
from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.state_control.common.estimators import MeanDifferenceEstimator
from aisteer360.algorithms.state_control.common.fit_specs import VectorTrainSpec
from aisteer360.algorithms.state_control.common.steering_vector import SteeringVector
from aisteer360.algorithms.state_control.caa.control import CAA

We use `ibm-granite/granite-4.1-3b`, a compact instruction-tuned model. Fitting runs one forward pass over each side of the contrastive data to read hidden states at every layer, so a GPU with enough memory for the model is recommended.

In [5]:
MODEL_NAME = "ibm-granite/granite-4.1-3b"

In [6]:
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

from tabulate import tabulate

## Contrastive data

CAA fits its direction from paired texts that differ only in the target behavior. Each pair here shares one user prompt, an everyday request that leaves the register free, and contrasts a formal assistant completion (complete words, measured phrasing, no exclamations) against a casual one (contractions, interjections, colloquial word choice). The two completions of a pair give the same substantive answer at comparable length, so the mean difference isolates the register rather than the content or a length cue, and topics vary across pairs for the same reason.

The formal pool is the positive side of the fit, which sets the sign convention for every control below, i.e., a positive `multiplier` moves the register towards formal and a negative one towards casual. The pools are hand-written and small for a self-contained demo; the persona vectors pipeline generates much larger pools automatically from a natural-language description of the trait.

In [7]:
formality_pairs = [
    {
        "prompt": "What's a good way to start learning to cook?",
        "formal": "I would suggest beginning with a few versatile techniques, such as roasting vegetables and "
                  "preparing a simple pan sauce. Repeating a small set of reliable dishes builds sound habits, "
                  "after which more ambitious recipes follow quite naturally.",
        "casual": "Honestly, just pick a couple of easy wins like roasted veggies or a simple pan sauce and make "
                  "them a bunch of times. Once you've got a few go-to dishes down, the fancier stuff isn't nearly "
                  "as scary.",
    },
    {
        "prompt": "Any tips for keeping houseplants alive?",
        "formal": "Most houseplants decline from excess water rather than neglect. Allow the soil to dry between "
                  "waterings, provide bright indirect light, and avoid relocating a plant once it has adjusted "
                  "to its position.",
        "casual": "Biggest tip: don't drown them! Let the soil dry out between waterings, stick them somewhere "
                  "bright but out of harsh sun, and once they're settled in a spot, just leave them alone.",
    },
    {
        "prompt": "How should I prepare for a long road trip?",
        "formal": "Have the vehicle serviced beforehand, plan fuel and rest stops at sensible intervals, and "
                  "assemble water, snacks, and a small emergency kit. Downloading offline maps is also advisable, "
                  "as coverage can lapse on rural stretches.",
        "casual": "Get the car checked out first, figure out roughly where you'll stop for gas and breaks, and "
                  "pack water, snacks, and a little emergency kit. Oh, and download offline maps, since signal "
                  "gets spotty out in the sticks.",
    },
    {
        "prompt": "What's a reasonable way to get back into running?",
        "formal": "Begin conservatively, alternating short intervals of running and walking three times per week, "
                  "and increase the total distance by roughly ten percent each week. Consistency at a modest "
                  "volume matters far more than any single ambitious session.",
        "casual": "Start small, like run-walk intervals three times a week, and bump the distance up maybe ten "
                  "percent a week. Showing up regularly beats one big heroic run every time, so don't overdo it "
                  "early!",
    },
    {
        "prompt": "My phone battery drains fast, what can I do?",
        "formal": "Review the battery usage report to identify demanding applications, reduce screen brightness, "
                  "and disable background refresh for applications that do not require it. If the battery is "
                  "several years old, a replacement may be the most effective remedy.",
        "casual": "Check the battery stats to see what's eating it, turn the brightness down, and switch off "
                  "background refresh for apps that don't need it. And if the battery's a few years old, "
                  "honestly, swapping it might be the real fix.",
    },
    {
        "prompt": "How do I make my mornings less chaotic?",
        "formal": "Prepare the evening before by laying out clothing, packing what you will need, and deciding "
                  "on breakfast in advance. A consistent waking time also helps considerably, since much of the "
                  "disorder stems from negotiating decisions while pressed for time.",
        "casual": "Do the prep the night before: lay out your clothes, pack your bag, know what breakfast is "
                  "gonna be. Waking up at the same time every day helps a ton too, 'cause most of the chaos is "
                  "just making decisions while you're rushed.",
    },
    {
        "prompt": "What should I know before adopting a cat?",
        "formal": "Budget for food, litter, and routine veterinary care, and prepare a quiet room where the cat "
                  "can acclimate during the first days. Adult cats often settle more readily than kittens, and "
                  "their temperament is already apparent at adoption.",
        "casual": "Plan on spending for food, litter, and vet visits, and set up a quiet room where the cat can "
                  "chill for the first few days. Grown-up cats often settle in easier than kittens, plus you "
                  "already know what they're like!",
    },
    {
        "prompt": "How can I get better at remembering names?",
        "formal": "Repeat the name immediately upon introduction, use it once or twice in conversation, and "
                  "associate it with a distinctive feature or context. Writing the name down shortly afterwards "
                  "further strengthens the association.",
        "casual": "Say the name back right when you meet them, drop it into the convo once or twice, and tie it "
                  "to something memorable about them. It'll stick way better if you jot it down after, too.",
    },
    {
        "prompt": "What's a good approach to decluttering?",
        "formal": "Proceed one category at a time rather than one room at a time, retain the items you use or "
                  "genuinely value, and remove discarded items from the house promptly. Short, regular sessions "
                  "tend to be more sustainable than a single exhausting purge.",
        "casual": "Go one category at a time instead of room by room, keep the stuff you actually use or love, "
                  "and don't let the discard pile hang around, get it out of the house fast. A bunch of short "
                  "sessions beats one giant exhausting purge, trust me.",
    },
    {
        "prompt": "How do I brew better coffee at home?",
        "formal": "Purchase whole beans, grind them immediately before brewing, and weigh both the coffee and "
                  "the water; a ratio near one to sixteen is a dependable starting point. Water just off the "
                  "boil extracts more evenly than water at a full boil.",
        "casual": "Get whole beans and grind them right before you brew, and weigh your coffee and water, "
                  "something like one to sixteen is a solid start. And don't pour the water right at a boil; "
                  "let it sit a sec first, it makes a real difference.",
    },
    {
        "prompt": "Any tips for writing a short bio about myself?",
        "formal": "Write in the third person, lead with your current role, add one or two notable "
                  "accomplishments, and close with a brief personal detail. Aim for three or four sentences and "
                  "set the draft aside before a final revision.",
        "casual": "Write it in the third person, kick off with what you do now, toss in an accomplishment or "
                  "two, and end with one fun personal bit. Keep it to three or four sentences, and don't do "
                  "the final pass until you've slept on it.",
    },
    {
        "prompt": "How do I keep bananas from ripening too fast?",
        "formal": "Separate the bananas from the bunch, keep them away from other fruit, and wrap each stem in "
                  "plastic to slow the release of ethylene. Once they reach the desired ripeness, refrigeration "
                  "halts the process, although the peel will darken.",
        "casual": "Split them off the bunch, keep them away from other fruit, and wrap the stems in a bit of "
                  "plastic; that slows down the ethylene. Once they're ripe enough, chuck them in the fridge. "
                  "The peel goes dark but the inside's fine.",
    },
    {
        "prompt": "What's a sensible way to back up my photos?",
        "formal": "Follow the three-two-one principle: three copies of the data, on two different types of "
                  "storage, with one copy kept off site. In practice, an automatic cloud backup combined with a "
                  "periodic copy to an external drive satisfies this comfortably.",
        "casual": "Go with the three-two-one thing: three copies, two kinds of storage, one of them off site. "
                  "Basically, let a cloud service back stuff up automatically and copy everything to an external "
                  "drive every so often, and you're covered.",
    },
    {
        "prompt": "How can I make small talk less awkward?",
        "formal": "Ask open questions about the immediate context, listen for details worth pursuing, and offer "
                  "small observations of your own so the exchange does not resemble an interview. Brief silences "
                  "are normal and rarely as noticeable as they feel.",
        "casual": "Ask open questions about whatever's going on around you, actually listen for threads to pull "
                  "on, and share little things yourself so it doesn't turn into an interview. And don't sweat "
                  "the pauses; nobody notices them like you do.",
    },
    {
        "prompt": "What stretches help after sitting all day?",
        "formal": "Prioritize the areas that shorten while seated: the hip flexors, the chest, and the "
                  "hamstrings. A kneeling hip flexor stretch, a doorway chest stretch, and a standing hamstring "
                  "stretch, each held for thirty seconds or so, cover these well.",
        "casual": "Hit the spots that get tight from sitting: hips, chest, hamstrings. A kneeling hip flexor "
                  "stretch, a doorway chest stretch, and a standing hamstring stretch, maybe thirty seconds "
                  "each, and you're pretty much sorted.",
    },
    {
        "prompt": "How do I pick a good watermelon?",
        "formal": "Choose a melon that feels heavy for its size, with a large creamy yellow patch where it "
                  "rested on the ground, and a dull rather than glossy rind. A deep hollow sound when tapped is "
                  "a further favorable indication.",
        "casual": "Grab one that feels heavy for its size, look for a big creamy yellow spot where it sat on "
                  "the ground, and skip the shiny ones, they're usually underripe. If it sounds nice and "
                  "hollow when you knock on it, even better!",
    },
    {
        "prompt": "Any advice for hosting dinner for the first time?",
        "formal": "Choose a dish you have prepared successfully before, complete whatever can be done in "
                  "advance, and set the table early. Guests take their cue from the host, so a composed welcome "
                  "matters more than an elaborate menu.",
        "casual": "Make something you've cooked before and know turns out fine, prep whatever you can ahead of "
                  "time, and set the table early. People mostly vibe off the host, so if you're relaxed, nobody "
                  "cares how fancy the food is.",
    },
    {
        "prompt": "How do I stop hitting snooze in the morning?",
        "formal": "Place the alarm across the room so that rising is required to silence it, keep a consistent "
                  "sleep schedule, and seek bright light promptly upon waking. An earlier bedtime addresses the "
                  "underlying cause more directly than any alarm strategy.",
        "casual": "Stick your alarm across the room so you have to get up to shut it off, keep your sleep "
                  "schedule steady, and get some bright light going as soon as you're up. And honestly, going "
                  "to bed earlier fixes the actual problem.",
    },
]

train_pairs = ContrastivePairs(
    prompts=[pair["prompt"] for pair in formality_pairs],
    positives=[pair["formal"] for pair in formality_pairs],
    negatives=[pair["casual"] for pair in formality_pairs],
)

print(f"{len(train_pairs.positives)} contrastive pairs")


18 contrastive pairs


We hold out a handful of prompts for evaluation. None of them appear in the fitting data, and each is an ordinary request that a chat model can answer in either register, so the register of the response is free to move under steering.

In [8]:
eval_prompts = [
    "Why do we get songs stuck in our heads?",
    "Give me some tips for my first job interview next week.",
    "What should I cook for a quick weeknight dinner?",
    "My laptop has gotten slow, what can I do about it?",
    "How does sourdough bread rise without commercial yeast?",
    "How do I get better at waking up early?",
]

## Baseline behavior

We load the model and tokenizer once and share them across every pipeline in this notebook by passing the loaded objects to each `SteeringPipeline` as `model=` and `tokenizer=`, so the pipelines differ only in their controls. Generation runs through `pipeline.generate(messages=...)` everywhere, which applies the chat template, generates, and returns only the completion; the baseline uses a control-free pipeline so that a single code path serves the whole notebook.

In [9]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = model.device

gen_params = {
    "max_new_tokens": 120,
    "do_sample": False,
    "repetition_penalty": 1.1,
    "pad_token_id": tokenizer.eos_token_id,
}

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:08<00:08,  8.68s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  5.59s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.06s/it]

In [10]:
baseline_pipeline = SteeringPipeline(model=model, tokenizer=tokenizer)
baseline_pipeline.steer()

baseline_responses = baseline_pipeline.generate(
    messages=[[{"role": "user", "content": prompt}] for prompt in eval_prompts],
    **gen_params,
)

print(tabulate(
    [[prompt, response] for prompt, response in zip(eval_prompts, baseline_responses)],
    headers=["prompt", "baseline response"],
    tablefmt="grid",
    maxcolwidths=[28, 84],
))

+------------------------------+--------------------------------------------------------------------------------------+
| prompt                       | baseline response                                                                    |
+==============================+======================================================================================+
| Why do we get songs stuck in | Songs getting stuck in your head, commonly referred to as “earworms,” occur due to a |
| our heads?                   | variety of psychological and neurological factors:                                   |
|                              | 1. **Repetition and Memorability**: Songs that are catchy, repetitive, or have a     |
|                              | strong emotional impact are more likely to be remembered. The repetition helps       |
|                              | encode the music into long-term memory.                                              |
|                              | 2. **Em

## Fitting the formality direction

`MeanDifferenceEstimator` renders each pair through the model's chat template (`prompt_format="chat_completion"` renders the prompt as a user turn and appends the completion after the generation prompt), runs one forward pass over each side, and takes the mean difference between the formal and casual activations at every layer. The result is a `SteeringVector` holding one direction per layer.

The `accumulate="last_token"` argument reads each example's hidden state at the final completion token. This is the extraction point of the original CAA setup, where each completion is a one-token multiple-choice answer; with full responses the final token comes after the model has processed the entire completion, so its state summarizes the response's register.

Note that passing `data=` and `train_spec=` to `CAA` runs this same fit inside `steer()`. We fit the vector standalone here so that one fit serves every steering configuration below and can be saved for the serving section.

In [11]:
train_spec = VectorTrainSpec(
    method="mean_diff",
    accumulate="last_token",
    prompt_format="chat_completion",
)

formality_vector = MeanDifferenceEstimator().fit(
    model,
    tokenizer,
    data=train_pairs,
    spec=train_spec,
)

print(f"Fitted a formality direction for {len(formality_vector.directions)} layers")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Fitted a formality direction for 40 layers


## Steering towards a formal register

`CAA` adds `multiplier * v` to the residual stream at the output of one layer. The formal pool is the positive side of the fit, so a positive `multiplier` moves the register towards formal. We add at a mid-depth layer; if `layer_id` is omitted the control selects a layer at roughly 40 percent depth, the depth range the CAA paper found most effective. The default `token_scope="after_prompt"` applies the vector at generated positions only, leaving the prompt's own representation untouched. `use_norm_preservation=True` rescales steered hidden states whose norm increased back to their pre-steering norm.

Note that useful values of `multiplier` depend on the model, the layer, and the accumulation mode, so the value below is a starting point worth sweeping. Values that are too small change little, and values that are too large degrade fluency, which surfaces as repetition, garbled phrasing, or drift away from the question.

In [12]:
LAYER_ID = 16
MULTIPLIER = 4.0

caa_formal = CAA(
    steering_vector=formality_vector,
    layer_id=LAYER_ID,
    multiplier=MULTIPLIER,
    use_norm_preservation=True,
)

formal_pipeline = SteeringPipeline(controls=[caa_formal], model=model, tokenizer=tokenizer)
formal_pipeline.steer()

formal_responses = formal_pipeline.generate(
    messages=[[{"role": "user", "content": prompt}] for prompt in eval_prompts],
    **gen_params,
)

## Steering towards a casual register

Flipping the sign subtracts the direction, moving the register towards the casual end of the same axis. One fitted vector therefore serves both ends of the dimension, and the two steered configurations differ only in the sign of `multiplier`. Movement in both directions along a single vector also serves as the causal check that the direction captures the register rather than an artifact of either pool.

In [13]:
caa_casual = CAA(
    steering_vector=formality_vector,
    layer_id=LAYER_ID,
    multiplier=-MULTIPLIER,
    use_norm_preservation=True,
)

casual_pipeline = SteeringPipeline(controls=[caa_casual], model=model, tokenizer=tokenizer)
casual_pipeline.steer()

casual_responses = casual_pipeline.generate(
    messages=[[{"role": "user", "content": prompt}] for prompt in eval_prompts],
    **gen_params,
)

## Comparing responses

The table places the three configurations side by side on each held-out prompt. The relevant reading is the phrasing rather than the substance of the advice, and since decoding is greedy, differences between the columns come from the steering alone.

In [14]:
table_rows = []
for i, prompt in enumerate(eval_prompts):
    table_rows.append([prompt, baseline_responses[i], casual_responses[i], formal_responses[i]])

print(tabulate(
    table_rows,
    headers=["prompt", "baseline", f"multiplier={-MULTIPLIER}", f"multiplier={MULTIPLIER}"],
    tablefmt="grid",
    maxcolwidths=[28, 42, 42, 42],
))

+------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+
| prompt                       | baseline                                   | multiplier=-4.0                            | multiplier=4.0                             |
+==============================+============================================+============================================+============================================+
| Why do we get songs stuck in | Songs getting stuck in your head, commonly | Songs getting stuck in your head is a      | Songs getting stuck in the mind, commonly  |
| our heads?                   | referred to as “earworms,” occur due to a  | common experience, and there's actually    | referred to as “earworms,” are thought to  |
|                              | variety of psychological and neurological  | some science behind it. This phenomenon    | result from the repetition of musical

## Saving and reloading the steering vector

The fitted `SteeringVector` is the reusable artifact of the whole procedure. The `save()` method writes the per-layer directions along with provenance metadata (model, tokenizer, and chat template fingerprints) to a `.svec` JSON file, and `SteeringVector.load()` restores it. Everything else, i.e., `layer_id`, `multiplier`, `token_scope`, and `use_norm_preservation`, is a per-control parameter set at construction time, so a saved vector supports new steering configurations without refitting.

To confirm the round trip, we reload the file into a fresh control at a smaller `multiplier` and run it on the Hugging Face backend, reusing the model already in memory.

In [15]:
os.makedirs("tmp", exist_ok=True)

VECTOR_PATH = "tmp/formality_vector.svec"

formality_vector.save(VECTOR_PATH)
reloaded_vector = SteeringVector.load(VECTOR_PATH)

caa_reloaded = CAA(
    steering_vector=reloaded_vector,
    layer_id=LAYER_ID,
    multiplier=2.0,
    use_norm_preservation=True,
)

reloaded_pipeline = SteeringPipeline(controls=[caa_reloaded], model=model, tokenizer=tokenizer)
reloaded_pipeline.steer()

response = reloaded_pipeline.generate(
    messages=[{"role": "user", "content": eval_prompts[0]}],
    **gen_params,
)
print(response)

Songs getting “stuck” in a person’s mind, commonly referred to as “earworms,” involve the involuntary recurrence of musical fragments or lyrical phrases without an apparent external trigger. Several theories attempt to explain this phenomenon:

1. **Memory and Retrieval**: The human memory system is capable of retaining vast amounts of information, including auditory data. When a song is repeatedly exposed to an individual—through listening, hearing snippets on television, radio, or other media—the neural pathways associated with that music become strengthened. This strengthening facilitates rapid retrieval of the musical content when it enters conscious thought.

2


## Serving through a vLLM server

The additive intervention `CAA` performs has a wire form. On a vLLM backend the pipeline registers no torch hooks; instead it serializes the control's configuration into an intervention spec, ships the direction tensor as a content-addressed artifact, and the [vLLM-Hook](https://github.com/IBM/vLLM-Hook) plugin applies the same edit inside the engine. The `vllm-serve` backend targets a running vLLM server through its OpenAI-compatible endpoints and needs no vLLM installation on the client.

The server environment carries the model and the plugin, i.e., `vllm` and the `vllm_hook_plugins` package (see the [vLLM-Hook](https://github.com/IBM/vLLM-Hook) repository) are installed there, and the server starts with `VLLM_HOOK_WORKER=unified` and eager execution:

```bash
VLLM_HOOK_WORKER=unified vllm serve ibm-granite/granite-4.1-3b --port 8000 --enforce-eager
```

The `artifact_dir` option names the directory the client writes tensors into, and it must be the same directory the server's registry reads, i.e., the server's `VLLM_HOOK_REGISTRY_DIR`, on a filesystem both sides can see (without the option, the client instead PUTs each artifact to the plugin's HTTP artifact route and no directory agreement is needed). This notebook illustrates the flow locally, i.e., the cells below start the same server as a subprocess on this machine, `base_url` points at localhost, and `artifact_dir` is a folder under `tmp/`. Note that in practice none of this process management exists on the client since the server runs on a separate GPU box with the model and plugin loaded there; the client sets only `base_url` and the shared `artifact_dir`.

The next cell checks that the `vllm` CLI and the `vllm_hook_plugins` package are present (both are installed by the toolkit's `vllm` extra) and that the allocated GPU can host a second CUDA process next to the kernel. The GPU hosts two processes only in its default (shared) compute mode or with MPS active; under exclusive-process mode without MPS the server exits with a device-unavailable error before serving anything. Note that on a managed cluster the compute mode is a property of the job request, e.g., LSF's `-gpu "num=1:mode=shared:j_exclusive=yes"` or `-gpu "num=1:mode=exclusive_process:mps=yes"` where policy pins the mode.

In [16]:
import atexit
import importlib.util
import shutil
import signal
import socket
import subprocess
import time
import urllib.error
import urllib.request

if shutil.which("vllm") is None or importlib.util.find_spec("vllm_hook_plugins") is None:
    raise RuntimeError(
        "the vllm CLI and the vllm_hook_plugins package are required; install the toolkit's vllm extra"
    )

mode_query = ["nvidia-smi", "--query-gpu=compute_mode", "--format=csv,noheader"]
visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
if visible_devices:
    mode_query += ["-i", visible_devices]
try:
    mode_output = subprocess.run(mode_query, capture_output=True, text=True).stdout
    compute_modes = [line.strip() for line in mode_output.splitlines() if line.strip()]
    mps_active = subprocess.run(["pgrep", "-f", "nvidia-cuda-mps"], capture_output=True).returncode == 0
except OSError:
    compute_modes, mps_active = [], False
if any(mode != "Default" for mode in compute_modes) and not mps_active:
    raise RuntimeError(
        f"GPU compute mode is {compute_modes} and MPS is not active; "
        "request the GPU in shared compute mode or with MPS"
    )

We start the server as a subprocess. The port is chosen dynamically so a stale server from an earlier run cannot answer the health checks below, `--gpu-memory-utilization 0.4` leaves room for the copy of the model this notebook already holds (`torch.cuda.empty_cache()` returns the kernel's cached blocks first), and the server log is written to `tmp/vllm_server.log`. The server's registry root is pointed at the same `tmp/vllm_artifacts` directory the spec below names as `artifact_dir` (via `VLLM_HOOK_REGISTRY_DIR`), which is the agreement the shared_fs transport requires. The subprocess starts in its own process group so that shutdown reaches the engine workers.

In [17]:
torch.cuda.empty_cache()

with socket.socket() as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    SERVER_PORT = port_probe.getsockname()[1]
SERVER_URL = f"http://localhost:{SERVER_PORT}"
SERVER_LOG_PATH = "tmp/vllm_server.log"
ARTIFACT_REGISTRY_DIR = os.path.abspath("tmp/vllm_artifacts")

server_command = [
    "vllm", "serve", MODEL_NAME,
    "--port", str(SERVER_PORT),
    "--enforce-eager",
    "--gpu-memory-utilization", "0.4",
]
server_env = {
    **os.environ,
    "VLLM_HOOK_WORKER": "unified",
    "VLLM_HOOK_REGISTRY_DIR": ARTIFACT_REGISTRY_DIR,
}
server_log = open(SERVER_LOG_PATH, "w")
server_process = subprocess.Popen(
    server_command,
    env=server_env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

A failure in a later cell must not leave the engine holding the GPU, so `stop_server` terminates the server's process group (falling back to a kill when termination stalls) and closes the log. Registering it with `atexit` covers kernel exit; the teardown cell at the end of the section calls the same function.

In [18]:
@atexit.register
def stop_server() -> None:
    if server_process.poll() is None:
        try:
            os.killpg(server_process.pid, signal.SIGTERM)
            server_process.wait(timeout=60)
        except ProcessLookupError:
            pass
        except subprocess.TimeoutExpired:
            os.killpg(server_process.pid, signal.SIGKILL)
            server_process.wait(timeout=10)
    if not server_log.closed:
        server_log.close()

We wait until the server answers `/version` (the endpoint the backend probes on construction) and then `/v1/hook/capabilities` (the discovery surface the backend reads next), so a broken or absent plugin fails here rather than inside `steer()`. The wait allows up to thirty minutes for engine boot and weight load; on failure the cell prints the tail of the server log before raising.

In [19]:
failure = None
for _ in range(360):
    if server_process.poll() is not None:
        failure = "vLLM server exited during startup"
        break
    try:
        urllib.request.urlopen(f"{SERVER_URL}/version", timeout=5)
        break
    except OSError:
        time.sleep(5)
else:
    failure = "vLLM server did not come up in time"

if failure is None:
    try:
        urllib.request.urlopen(f"{SERVER_URL}/v1/hook/capabilities", timeout=30)
    except urllib.error.HTTPError as error:
        print(error.read().decode(errors="replace")[:2000])
        failure = f"hook discovery route answered HTTP {error.code}"
    except OSError as error:
        failure = f"hook discovery route unreachable: {error}"

if failure is not None:
    server_log.flush()
    with open(SERVER_LOG_PATH, errors="replace") as log_file:
        print("".join(log_file.readlines()[-40:]))
    stop_server()
    raise RuntimeError(f"{failure}; the tail of {SERVER_LOG_PATH} is printed above")

print(f"server is up at {SERVER_URL}")

server is up at http://localhost:60449


Note that the client never loads model weights. With a precomputed vector, `CAA`'s steer step needs only structural facts about the model (the layer count), which the pipeline reads through the server session, so the pipeline is constructed with no local model. `steer()` checks support before any work happens; a configuration with no wire form, or a server without the plugin, raises with a verdict naming the gap. Using the pipeline as a context manager releases the client's backend on exit. The offline engine (`BackendSpec(kind="vllm")`) is the in-process alternative where the pipeline boots and releases the engine itself.

The `multiplier` remains a per-deployment choice set after loading, so the served configuration below sets its own value. Also note that on API backends the generation parameter table is exhaustive, so `model.generate` extras such as `pad_token_id` raise rather than pass through; the call below therefore names its parameters explicitly instead of reusing `gen_params`.

In [20]:
SERVE_MULTIPLIER = 3.0

serve_spec = BackendSpec(
    kind="vllm-serve",
    model=MODEL_NAME,
    options={
        "base_url": SERVER_URL,
        "hook_plugin": True,
        "artifact_dir": ARTIFACT_REGISTRY_DIR,
    },
)

caa_served = CAA(
    steering_vector=SteeringVector.load(VECTOR_PATH),
    layer_id=LAYER_ID,
    multiplier=SERVE_MULTIPLIER,
    use_norm_preservation=True,
)

with SteeringPipeline(controls=[caa_served], backend=serve_spec) as served_pipeline:
    served_pipeline.steer()
    served_responses = served_pipeline.generate(
        messages=[[{"role": "user", "content": prompt}] for prompt in eval_prompts[:2]],
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.1,
    )

served_rows = []
for prompt, response in zip(eval_prompts[:2], served_responses):
    served_rows.append([prompt, response])

print(tabulate(
    served_rows,
    headers=["prompt", f"served (multiplier={SERVE_MULTIPLIER})"],
    tablefmt="grid",
    maxcolwidths=[28, 84],
))

`torch_dtype` is deprecated! Use `dtype` instead!


+------------------------------+--------------------------------------------------------------------------------------+
| prompt                       | served (multiplier=3.0)                                                              |
+==============================+======================================================================================+
| Why do we get songs stuck in | Songs getting stuck in our heads, commonly referred to as “earworms,” are a common   |
| our heads?                   | and intriguing phenomenon. Several factors contribute to the occurrence of earworms: |
|                              | 1. **Memory Encoding**: When a song is repeatedly heard, it becomes encoded into     |
|                              | long-term memory. The auditory information associated with the music is stored in    |
|                              | various regions of the brain, including the auditory cortex.                         |
|                              | 2. **At

The server sits outside the pipeline's lifecycle since a served engine is meant to outlive its clients, so we stop the subprocess and unregister the `atexit` hook explicitly.

In [21]:
stop_server()
atexit.unregister(stop_server)

## Summary

This notebook fitted a formality direction for `ibm-granite/granite-4.1-3b` as the mean difference between hidden states on formal and casual completions of shared prompts (the contrastive-response setup used for persona vectors), read at each completion's final token. Adding the direction at a single mid-depth layer moves held-out responses towards complete words and measured phrasing, and subtracting it moves them towards contractions and colloquial word choice.

The fitted `SteeringVector` round-trips through `save()` and `load()`, and `layer_id`, `multiplier`, `token_scope`, and `use_norm_preservation` are per-control construction parameters, so one fit serves any steering configuration. The same artifact ran in process on the Hugging Face backend and through a vLLM server, where the pipeline lowers the control to an intervention spec executed by the vLLM-Hook plugin; support is checked before any work happens, and the client holds no model weights.

The recipe generalizes by swapping the data. Any persona dimension expressible as paired completions over shared prompts can be fitted the same way (the persona vectors paper automates exactly this pair generation), and a systematic sweep over multipliers or layers belongs in a `Benchmark` via `ControlSpec.vars` (see the benchmark notebooks).